# Do dado sintético ao modelo em produção 🤖

# BLOCO 1 — Dados sintéticos com sabor de domínio

## Exercício 1.1 — Seu primeiro dataset sintético

In [ ]:
!pip install scikit-learn joblib pandas numpy huggingface_hub fastapi uvicorn

In [ ]:
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    class_sep=1.0,
    weights=[0.7, 0.3],
    random_state=42
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y

print(df.head())
print("\nDistribuição do target:")
print(df["target"].value_counts())

# class_sep=0.5 torna o problema mais difícil porque as classes se misturam mais.
# class_sep=3.0 torna o problema muito fácil porque as classes ficam mais separadas.


# Sem random_state fixo, você não sabe se um modelo melhorou de verdade.

   feature_0  feature_1  feature_2  feature_3  feature_4  target
0  -0.038769  -0.649239  -0.224746  -1.346275   0.126879       0
1   1.005284  -1.373239   1.157346   0.126493   1.422799       0
2  -0.742455  -0.573257   1.688442  -2.588237   0.762562       0
3  -1.587158   1.758582  -0.930664   0.764614   2.415399       1
4  -0.941758   0.367913  -0.549360  -2.029919  -1.503957       0

Distribuição do target:
target
0    699
1    301
Name: count, dtype: int64


## Exercício 1.2 — Adicionando sabor de domínio

In [ ]:
import numpy as np
import pandas as pd

def gerar_dataset(n_samples: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    fraude = rng.integers(0, 2, size=n_samples)

    valor_transacao = np.where(
        fraude,
        rng.uniform(500, 10000, n_samples),
        rng.uniform(10, 800, n_samples)
    ).round(2)

    hora_transacao = np.where(
        fraude,
        rng.integers(0, 6, n_samples),
        rng.integers(7, 23, n_samples)
    )

    distancia_ultima_compra = np.where(
        fraude,
        rng.uniform(100, 5000, n_samples),
        rng.uniform(0, 50, n_samples)
    ).round(1)

    tentativas_senha = np.where(
        fraude,
        rng.integers(2, 10, n_samples),
        rng.integers(1, 2, n_samples)
    )

    pais_diferente = (rng.random(n_samples) < np.where(fraude, 0.4, 0.05)).astype(int)

    df = pd.DataFrame({
        "valor_transacao": valor_transacao,
        "hora_transacao": hora_transacao,
        "distancia_ultima_compra": distancia_ultima_compra,
        "tentativas_senha": tentativas_senha,
        "pais_diferente": pais_diferente,
        "target": fraude
    })

    return df

df = gerar_dataset(n_samples=2000, seed=42)
df.groupby("target").mean().round(2)

,valor_transacao,hora_transacao,distancia_ultima_compra,tentativas_senha,pais_diferente
target,,,,,
0,399.59,14.41,25.61,1.00,0.05
1,5264.42,2.50,2567.97,5.61,0.40


## Exercício 1.3 — Desafio: função geradora robusta 🎯

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple

def gerar_dataset(
    n_samples: int = 1000,
    seed: int = 42,
    proporcao_positivos: float = 0.3
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Gera dataset sintético para detecção de fraude.

    Parâmetros
    ----------
    n_samples : int
        Número de amostras.
    seed : int
        Seed para reprodutibilidade.
    proporcao_positivos : float
        Proporção da classe positiva entre 0.05 e 0.95.

    Retorna
    -------
    df : pd.DataFrame
        Dataset completo.
    X : np.ndarray
        Matriz de features.
    y : np.ndarray
        Vetor target.

    Exemplo
    -------
    >>> df, X, y = gerar_dataset(500, 42, 0.3)
    """

    if not (0.05 <= proporcao_positivos <= 0.95):
        raise ValueError("proporcao_positivos deve estar entre 0.05 e 0.95")

    rng = np.random.default_rng(seed)
    fraude = rng.choice([0, 1], size=n_samples, p=[1 - proporcao_positivos, proporcao_positivos])

    valor_transacao = np.where(
        fraude,
        rng.uniform(500, 10000, n_samples),
        rng.uniform(10, 800, n_samples)
    ).round(2)

    hora_transacao = np.where(
        fraude,
        rng.integers(0, 6, n_samples),
        rng.integers(7, 23, n_samples)
    )

    distancia_ultima_compra = np.where(
        fraude,
        rng.uniform(100, 5000, n_samples),
        rng.uniform(0, 50, n_samples)
    ).round(1)

    tentativas_senha = np.where(
        fraude,
        rng.integers(2, 10, n_samples),
        rng.integers(1, 2, n_samples)
    )

    pais_diferente = (rng.random(n_samples) < np.where(fraude, 0.4, 0.05)).astype(int)

    df = pd.DataFrame({
        "valor_transacao": valor_transacao,
        "hora_transacao": hora_transacao,
        "distancia_ultima_compra": distancia_ultima_compra,
        "tentativas_senha": tentativas_senha,
        "pais_diferente": pais_diferente,
        "target": fraude
    })

    X = df.drop(columns=["target"]).values
    y = df["target"].values

    return df, X, y

df, X, y = gerar_dataset(n_samples=1000, seed=42, proporcao_positivos=0.3)
print(df.head())
print(X.shape, y.shape)

   valor_transacao  hora_transacao  distancia_ultima_compra  tentativas_senha  \
0          1089.60               3                   3608.9                 6   
1           362.65              13                     33.1                 1   
2          1725.79               3                   4274.2                 2   
3           524.13              16                      8.3                 1   
4           101.56              13                      2.2                 1   

   pais_diferente  target  
0               0       1  
1               0       0  
2               0       1  
3               0       0  
4               0       0  
(1000, 5) (1000,)


# BLOCO 2 — Treinar e serializar o modelo

## Exercício 2.1 — Treinar e avaliar

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df, X, y = gerar_dataset(n_samples=2000, seed=42, proporcao_positivos=0.3)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred, target_names=["legitimo", "fraude"])
print(report)

              precision    recall  f1-score   support

    legitimo       1.00      1.00      1.00       276
      fraude       1.00      1.00      1.00       124

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



O recall da classe fraude é mais importante que a precision, porque detectar as fraudes que reduz o risco de prejuízo.

## Exercício 2.2 — Serializar o artefato

In [ ]:
import joblib
import numpy as np
import os

joblib.dump(model, "model.pkl")
tamanho_kb = os.path.getsize("model.pkl") / 1024
print(f"Modelo salvo em model.pkl ({tamanho_kb:.2f} KB)")

model_carregado = joblib.load("model.pkl")

amostra = X_test[:5]
pred_original = model.predict(amostra)
pred_carregado = model_carregado.predict(amostra)

assert np.array_equal(pred_original, pred_carregado), "As predições divergem!"
print("Artefato validado com sucesso")
print("Predições:", pred_carregado)

Modelo salvo em model.pkl (63.77 KB)
Artefato validado com sucesso
Predições: [0 1 0 1 0]


# BLOCO 3 — Hugging Face Hub como registry de modelos

## Exercício 3.1 — Autenticar e criar o repositório

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/mlops-fraude-v1"

repo_url = api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
    private=False
)

print("Usuário:", username)
print("Repositório:", repo_url)

## Exercício 3.2 — Escrever o model card

In [ ]:
---
language: pt
tags:
  - sklearn
  - classification
  - fraud-detection
  - mlops
---

### mlops-fraude-v1

Modelo de classificação binária para detecção de transações potencialmente fraudulentas.

#### Objetivo

Este modelo foi treinado com dados sintéticos para fins educacionais, simulando um cenário de detecção de fraude em transações financeiras.

#### Features de entrada

- valor_transacao
- hora_transacao
- distancia_ultima_compra
- tentativas_senha
- pais_diferente

#### Métricas

- Precision (fraude): 0.xx
- Recall (fraude): 0.xx
- F1-score (fraude): 0.xx

#### Limitações

- O dataset é sintético e simplificado.
- As distribuições não representam todas as complexidades do mundo real.
- O modelo não foi calibrado para produção real.

#### Frase para responder
 Um model card é importante porque documenta o que o modelo faz, como foi treinado, quais entradas espera e quais limitações possui. Mesmo em um modelo simples, isso melhora rastreabilidade e entendimento do artefato.

#### Uso

```python
from huggingface_hub import hf_hub_download
import joblib

path = hf_hub_download(repo_id="SEU_USUARIO/mlops-fraude-v1", filename="model.pkl")
model = joblib.load(path)




## Exercício 3.3 — Publicar

In [ ]:
from huggingface_hub import HfApi
import sklearn
import joblib as jl
import numpy as np

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/mlops-fraud-v1"

requirements = (
    f"scikit-learn=={sklearn.__version__}\n"
    f"joblib=={jl.__version__}\n"
    f"numpy=={np.__version__}\n"
)

with open("requirements.txt", "w") as f:
    f.write(requirements)

for filename in ["model.pkl", "README.md", "requirements.txt"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"chore: add {filename}"
    )
    print(f"✅ {filename} publicado")

print(f"\n🔗 https://huggingface.co/{repo_id}")

# Quando um novo model.pkl é publicado,
# o repositório passa a ter uma nova versão do artefato. Para saber qual versão está em produção,
# é importante registrar o commit.

# BLOCO 4 — Carregar o modelo e integrar à API

## Exercício 4.1 — A função `load_model`

In [ ]:
from huggingface_hub import hf_hub_download
import joblib
import time

def load_model(repo_id: str, filename: str = "model.pkl", force_download: bool = False):
    t0 = time.time()
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    model = joblib.load(local_path)
    elapsed = time.time() - t0
    origem = "Hub (forçado)" if force_download else "cache/local"
    print(f"Modelo carregado de {origem} em {elapsed:.3f}s")
    return model

REPO_ID = "SEU_USUARIO/mlops-fraude-v1"

print("Primeira chamada")
model = load_model(REPO_ID)

print("\nSegunda chamada")
model = load_model(REPO_ID)

print("\nTerceira chamada (forçada)")
model = load_model(REPO_ID, force_download=True)

## Exercício 4.2 — Endpoint `/predict` com modelo real

In [ ]:
from huggingface_hub import hf_hub_download
import joblib

def load_model(repo_id: str, filename: str = "model.pkl", force_download: bool = False):
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    model = joblib.load(local_path)
    return model

In [ ]:
from fastapi import APIRouter
from pydantic import BaseModel, Field
import numpy as np

router = APIRouter()

REPO_ID = "SEU_USUARIO/mlops-fraude-v1"
_model = None

def get_model():
    global _model
    if _model is None:
        from model_utils import load_model
        _model = load_model(REPO_ID)
    return _model

class PredictInput(BaseModel):
    valor_transacao: float = Field(gt=0)
    hora_transacao: int = Field(ge=0, le=23)
    distancia_ultima_compra: float = Field(ge=0)
    tentativas_senha: int = Field(ge=1)
    pais_diferente: int = Field(ge=0, le=1)

class PredictOutput(BaseModel):
    prediction: int
    probability: float
    label: str
    model_version: str

@router.post("/predict", response_model=PredictOutput)
async def predict(input: PredictInput):
    model = get_model()

    features = np.array([[
        input.valor_transacao,
        input.hora_transacao,
        input.distancia_ultima_compra,
        input.tentativas_senha,
        input.pais_diferente
    ]])

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])
    label = "fraude" if prediction == 1 else "legitimo"

    return PredictOutput(
        prediction=prediction,
        probability=round(probability, 4),
        label=label,
        model_version=REPO_ID
    )

In [ ]:
from fastapi import FastAPI
from routers import predict

app = FastAPI(
    title="API de Predição de Fraude",
    version="1.0.0"
)

app.include_router(predict.router, prefix="/ml", tags=["ML"])

@app.get("/")
async def root():
    return {"message": "API de ML no ar"}

## Exercício 4.3 — Endpoint `/health` com status do modelo

In [ ]:
@router.get("/health")
async def health():
    try:
        model = get_model()
        test_input = np.zeros((1, 5))
        model.predict(test_input)

        return {
            "api": "ok",
            "model": "ok",
            "model_repo": REPO_ID
        }
    except Exception as e:
        return {
            "api": "ok",
            "model": "degraded",
            "model_repo": str(e)
        }